[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab12_pandas_loading_selecting/lab12_student.ipynb)

# Lab 12 — pandas I: Loading and Selecting Data

### Digital Skills and Business Decisioning Using Python

Until now, every dataset in this course has been a list, a dictionary, or a NumPy array that you built by hand or read line by line. Real data usually arrives as a **table**: rows of records, columns of attributes, in a spreadsheet or a CSV file. **pandas** is the Python library for working with tables, and it's the main tool you'll use for the rest of the course and in your final assignment.

This lab loads the full clothing reviews dataset (all ~23,000 reviews, not the 600-review sample) and teaches you to look around in it: what's there, how to pick out the columns and rows you care about, and how to count and sort. The next two labs clean it and summarise it.

In this lab you'll learn to:

* create pandas' two core objects, the **Series** and the **DataFrame**,
* **load** a CSV file with `pd.read_csv()` and take a first look with `head()`, `shape`, `info()`, and `describe()`,
* **select** columns, and rows by position (`iloc`) or by label (`loc`),
* **filter** rows with conditions, exactly like NumPy masks,
* **sort** rows with `sort_values()`, and
* **count** categories with `value_counts()`.

### Part 1: Series and DataFrames

pandas is always imported as `pd`. Its simplest object is a **Series**: a one-dimensional array of values (built on NumPy) with a **label** for each position, called the **index**.

In [ ]:
import pandas as pd

ratings = pd.Series([5, 4, 5, 1, 3])
print(ratings)

The left column is the index (by default `0, 1, 2, ...`) and the right column holds the values. At the bottom, `dtype: int64` is the type of the values, just as in NumPy.

A Series behaves like a NumPy array: arithmetic is vectorized, and aggregations are methods.

In [ ]:
print(ratings.mean())
print(ratings * 2)
print(ratings[ratings >= 4])

The difference from NumPy is the index. It doesn't have to be numbers: build a Series from a **dictionary**, and the keys become the labels.

In [ ]:
prices = pd.Series({"dress": 148.0, "tank": 38.0, "jacket": 198.0})
print(prices)
print(prices["jacket"])

### DataFrames

A **DataFrame** is a table: several Series side by side, sharing one index. Each column is a Series with a name. One easy way to build one is from a **list of dictionaries**, the shape you've used since Lab 04 (one dictionary per record, the keys become the column names):

In [ ]:
records = [
    {"product": "dress", "rating": 5, "price": 148.0},
    {"product": "tank", "rating": 3, "price": 38.0},
    {"product": "jacket", "rating": 4, "price": 198.0},
]

small = pd.DataFrame(records)
small

Notice the last line is just `small`, with no `print()`. In a notebook, if the last line of a cell is a value, Colab displays it, and DataFrames are displayed as a nicely formatted table. (`print(small)` works too, but looks plainer.) From now on we'll usually let the notebook display DataFrames this way.

Every column of a DataFrame is a Series:

In [ ]:
print(small["price"])
print(small["price"].mean())

### Practice: Series and DataFrames

**Exercise 1: A Small Table (easy)**

Build a DataFrame called `stock` from the list of dictionaries below, display it, and print the **total** of the `units` column.

*Expected result:* `57`

In [ ]:
rows = [
    {"item": "sweater", "units": 12},
    {"item": "jeans", "units": 30},
    {"item": "blazer", "units": 15},
]

# TODO: build stock = pd.DataFrame(rows), display it, then print the sum of the units column

In [ ]:
# @solution
rows = [
    {"item": "sweater", "units": 12},
    {"item": "jeans", "units": 30},
    {"item": "blazer", "units": 15},
]

stock = pd.DataFrame(rows)
print(stock["units"].sum())
stock

---

### Part 2: Loading a Real Dataset

The reviews dataset is stored as a **CSV file** (comma-separated values): a plain text file with one row per line and commas between the columns. It's the most common format for sharing tables, and `pd.read_csv()` loads one straight into a DataFrame, from a file on disk **or** from a web address.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/JanNikolaidovic/python-text-analytics-lab/main/data/womens_clothing_ecommerce_reviews.csv"

df = pd.read_csv(DATA_URL)
df.head()

`df` is the conventional name for "the DataFrame I'm working on". **`.head()`** shows the first 5 rows (`.head(10)` for 10, and `.tail()` for the last ones). It's always the first thing to run after loading data: before any analysis, *look at it*.

Look closely at the first column, `Unnamed: 0`. It just counts `0, 1, 2, ...`, exactly like the index on the far left. Whoever saved this file also saved pandas' index as a column, and when there's no header name for it, pandas calls it `Unnamed: 0`. It carries no information, so tell `read_csv()` to use that first column **as the index**, with `index_col=0`:

In [ ]:
df = pd.read_csv(DATA_URL, index_col=0)
df.head()

Much better. You've already done your first bit of data cleaning, and it came from simply *looking*.

### How Big Is It? `shape`, `columns`, `dtypes`

In [ ]:
print(df.shape)
print(df.columns)

`df.shape` is `(rows, columns)`, exactly like a 2-D NumPy array: 23,486 reviews, 10 columns. `df.columns` lists the column names.

### The Overview: `info()`

**`.info()`** is the most useful single summary of a DataFrame: every column, how many **non-missing** values it has, and its type.

In [ ]:
df.info()

Three things to take from this:

1. **The types.** `int64` columns hold whole numbers. The text columns show as `object` (in pandas 2) or `str` (in pandas 3): both mean "text". Whichever you see depends on the pandas version installed in Colab, and it makes no difference in this course.
2. **Missing values.** The dataset has 23,486 rows, but `Title` has only about 19,700 non-null values, `Review Text` about 22,600, and the three `... Name` columns are missing a handful. Some customers gave a rating without writing a review or a title. You'll deal with these in Lab 13.
3. **Memory usage** at the bottom: the whole dataset is under 2 MB in memory, so it's small enough to handle comfortably in Colab.

Here's what each column means:

| Column | Meaning |
|---|---|
| `Clothing ID` | Which product was reviewed (a numeric code) |
| `Age` | The reviewer's age |
| `Title` | The review's title |
| `Review Text` | The review itself |
| `Rating` | Stars, from 1 (worst) to 5 (best) |
| `Recommended IND` | Whether the customer recommends the product: 1 = yes, 0 = no |
| `Positive Feedback Count` | How many other customers marked the review as helpful |
| `Division Name`, `Department Name`, `Class Name` | Product categories, from broad to narrow |

### Summary Statistics: `describe()`

**`.describe()`** computes the standard summary statistics for every numeric column at once: the count, mean, standard deviation, minimum, maximum, and the **quartiles** (25%, 50% = median, 75%).

In [ ]:
df.describe()

A few things jump out, if you read it column by column:

* **`Age`** ranges from 18 to 99, with a median of 41.
* **`Rating`**: the median is already 5 and the 25% quartile is 4. At least three quarters of the reviews give 4 or 5 stars.
* **`Positive Feedback Count`**: the median is 1, but the maximum is 122. Most reviews get almost no helpful votes, and a few get a lot.
* **`Clothing ID`** gets a mean and a standard deviation, but they're meaningless: it's an ID code that happens to be stored as a number. `describe()` can't know that; you have to.

For the text columns, `describe(exclude="number")` gives a different summary: how many distinct values each has (`unique`), the most common one (`top`), and how often it appears (`freq`).

In [ ]:
df.describe(exclude="number")

The most common title is `Love it!`, and there are 20 different product classes.

### Practice: A First Look

**Exercise 2: The Last Rows (easy)**

Display the **last 3** rows of `df`.

In [ ]:
# TODO: display the last 3 rows

In [ ]:
# @solution
df.tail(3)

**Exercise 3: Reading `describe()` (medium)**

Using the output of `df.describe()` above (no new code needed for the first part), answer in a comment: what is the **median** age of reviewers, and what share of reviews give a rating of **at least 4**? Then check your answer about ratings with a line of code: `(df["Rating"] >= 4).mean()` (a mask and `.mean()`, just like NumPy).

In [ ]:
# TODO: answer in comments, then check the share of ratings >= 4

In [ ]:
# @solution
# Median age: 41 (the 50% row of the Age column).
# The 25% quartile of Rating is 4, so at least 75% of reviews give 4 or 5 stars.
print((df["Rating"] >= 4).mean())

---

### Part 3: Selecting Columns

Square brackets with a column name return that column as a **Series**:

In [ ]:
df["Rating"].head()

Square brackets with a **list** of names (so, two pairs of brackets) return a smaller **DataFrame** with just those columns, in the order you list them:

In [ ]:
df[["Rating", "Age", "Department Name"]].head()

> **One pair of brackets gives a Series, two pairs give a DataFrame.** `df["Rating"]` is a Series (one column); `df[["Rating"]]` is a DataFrame that happens to have one column. Most of the time you want the Series.

You may see columns accessed with a dot, like `df.Rating`. That works for simple names, but not for names with spaces (`df.Review Text` is a syntax error), so this course always uses square brackets.

### Reading a Full Review

pandas shortens long text in its tables. To read one review in full, pick out a single value with `.loc[row, column]` (explained properly in the next part) and print it:

In [ ]:
print(df.loc[2, "Title"])
print(df.loc[2, "Review Text"])

---

### Part 4: Selecting Rows, `iloc` and `loc`

There are two ways to pick rows, and the difference matters:

* **`.iloc[...]`** selects by **position**: the 1st row, the 5th row, rows 10 to 20. The `i` stands for *integer position*. It works exactly like list indexing, including negative numbers and slices.
* **`.loc[...]`** selects by **label**: the row whose index label is `2`, the column named `"Rating"`.

In [ ]:
df.iloc[0]

A single row comes back as a Series, with the column names as its index. Slices give you a DataFrame:

In [ ]:
df.iloc[10:13]

With `.loc`, you can give rows **and** columns, separated by a comma: `df.loc[rows, columns]`.

In [ ]:
df.loc[10:12, ["Rating", "Title"]]

Look carefully: `df.iloc[10:13]` and `df.loc[10:12]` return the same three rows. That's because **`.loc` slices include the end label**, while `.iloc` slices (like list slices) stop *before* the end. It's one of pandas' few genuinely confusing rules.

Right now positions and labels happen to be the same numbers, so the two look interchangeable. They stop being the same the moment you filter or sort, which is the next part.

### Practice: Selecting

**Exercise 4: Pick the Columns (easy)**

Display the `Title`, `Rating`, and `Recommended IND` columns for the **first 5** rows. (Hint: select the columns with double brackets, then use `.head()`.)

In [ ]:
# TODO: show Title, Rating, and Recommended IND for the first 5 rows

In [ ]:
# @solution
df[["Title", "Rating", "Recommended IND"]].head()

**Exercise 5: One Value (easy)**

Print the **age** and the **rating** of the review with index label `100`, using `.loc`.

In [ ]:
# TODO: print the Age and Rating of the review labelled 100

In [ ]:
# @solution
print(df.loc[100, "Age"])
print(df.loc[100, "Rating"])

---

### Part 5: Filtering Rows With Conditions

Filtering works exactly like NumPy masks (Lab 11). A comparison on a column gives a Series of `True`/`False`, and putting that inside `df[...]` keeps only the `True` rows:

In [ ]:
one_star = df[df["Rating"] == 1]

print(one_star.shape)
one_star.head()

Read `df[df["Rating"] == 1]` as *"the rows of `df` where the rating is 1"*. There are 842 one-star reviews.

Now look at the index on the left: `61, 85, 96, ...`, not `0, 1, 2`. **Filtering keeps the original labels.** This is where `iloc` and `loc` part ways: `one_star.iloc[0]` is the first row of the filtered table (label 61), while `one_star.loc[0]` would raise a `KeyError`, because the review labelled `0` wasn't a one-star review and isn't in the table.

In [ ]:
print(one_star.iloc[0]["Review Text"])

### Combining Conditions

As in NumPy, combine conditions with `&` (and) and `|` (or), and put **parentheses around each one**:

In [ ]:
young_unhappy = df[(df["Age"] < 30) & (df["Rating"] <= 2)]
print(len(young_unhappy))

extreme = df[(df["Rating"] == 1) | (df["Rating"] == 5)]
print(len(extreme))

For "is the value one of these?", **`.isin()`** is neater than a chain of `|`:

In [ ]:
outer = df[df["Department Name"].isin(["Jackets", "Trend"])]
print(len(outer))
outer[["Department Name", "Class Name", "Rating"]].head()

### Counting and Shares

Because a condition is a Series of `True`/`False`, the NumPy tricks from Lab 11 work unchanged: `.sum()` counts the `True` values, `.mean()` gives their share.

In [ ]:
recommends = df["Recommended IND"] == 1

print(recommends.sum())
print(f"{recommends.mean():.1%} of reviewers recommend the product.")

(`:.1%` is another f-string format: it multiplies by 100, shows one decimal, and adds the `%` sign.)

### Practice: Filtering

**Exercise 6: Dresses Only (easy)**

Create a DataFrame `dresses` with only the reviews whose `Department Name` is `"Dresses"`, and print how many there are.

*Expected result:* `6319`

In [ ]:
# TODO: filter the Dresses department and print the number of rows

In [ ]:
# @solution
dresses = df[df["Department Name"] == "Dresses"]
print(len(dresses))

**Exercise 7: Happy but Not Recommending? (medium)**

Some customers give 4 or 5 stars and still say they would **not** recommend the product. Find those reviews (rating at least 4 **and** `Recommended IND` equal to 0), print how many there are, and print the text of the first one (use `.iloc[0]`).

*Expected result:* `193` reviews

In [ ]:
# TODO: filter rating >= 4 and Recommended IND == 0, print the count and the first review text

In [ ]:
# @solution
mixed = df[(df["Rating"] >= 4) & (df["Recommended IND"] == 0)]
print(len(mixed))
print(mixed.iloc[0]["Review Text"])

---

### Part 6: Sorting

**`.sort_values(column)`** sorts the rows by a column, smallest first; add `ascending=False` for largest first. Like `sorted()` in Lab 04, it returns a **new** DataFrame and leaves `df` unchanged.

Which reviews did other customers find most helpful?

In [ ]:
most_helpful = df.sort_values("Positive Feedback Count", ascending=False)
most_helpful[["Rating", "Positive Feedback Count", "Title"]].head(10)

Look at the ratings: the top 3 includes a 3-star and a 1-star review, even though fewer than 1 in 5 reviews has 3 stars or fewer. Do critical reviews attract more helpful votes *on average*? A top 10 can't answer that; it's a question for the grouping tools in Lab 14.

You can sort by **several** columns by passing a list. Rows are sorted by the first column, and ties are broken by the second:

In [ ]:
df.sort_values(["Rating", "Age"], ascending=[True, False])[["Rating", "Age", "Title"]].head()

The lowest ratings come first, and within each rating the **oldest** reviewers come first (`ascending` can take one `True`/`False` per column).

---

### Part 7: Counting Categories

**`.value_counts()`** is the pandas version of `Counter`: it counts how often each value appears in a column, sorted from most to least common.

In [ ]:
df["Department Name"].value_counts()

Two useful options:

* `normalize=True` gives **shares** instead of counts.
* `.sort_index()` afterwards sorts by the values themselves instead of by frequency, which is what you want for an ordered scale like ratings.

In [ ]:
df["Rating"].value_counts(normalize=True).sort_index()

More than half of all reviews are 5 stars, and only about 1 in 10 is 1 or 2 stars. You saw the same pattern in the 600-review sample; now it's confirmed on the full dataset.

Two more counting tools: **`.nunique()`** is the number of distinct values, and **`.unique()`** lists them.

In [ ]:
print(df["Clothing ID"].nunique())
print(df["Division Name"].unique())

Look closely at the division names. One of them is spelled **`Initmates`**. That's a typo in the original data, and there's also a `nan` in the list, which is how pandas shows a missing value. `value_counts()` hides missing values by default, but `.unique()` shows them. Both problems go on the cleaning list for Lab 13.

### Practice: Sorting and Counting

**Exercise 8: The Oldest Reviewers (easy)**

Display the `Age`, `Rating`, and `Title` of the **5 oldest** reviewers.

In [ ]:
# TODO: sort by Age (largest first) and show Age, Rating, Title for the top 5

In [ ]:
# @solution
df.sort_values("Age", ascending=False)[["Age", "Rating", "Title"]].head()

**Exercise 9: Class Counts (medium)**

Show the number of reviews in each `Class Name`, then show only the classes with **fewer than 100** reviews. (Hint: `value_counts()` returns a Series, and you can filter a Series with a condition, just like a DataFrame.)

Why might classes with very few reviews be a problem when comparing average ratings between classes? Answer in a comment.

In [ ]:
# TODO: count reviews per Class Name, then keep only the classes with fewer than 100

In [ ]:
# @solution
class_counts = df["Class Name"].value_counts()
print(class_counts)
print(class_counts[class_counts < 100])
# An average over 1 or 2 reviews says almost nothing: a single unhappy customer
# can make a class look like the worst in the store.

---

### Recap: Your pandas Toolkit, Part 1

| Tool | What It Does | Example |
|---|---|---|
| `import pandas as pd` | Import pandas under its standard name | |
| `pd.Series(...)` | A labelled 1-D column of values | `pd.Series([5, 4, 1])` |
| `pd.DataFrame(...)` | A table (e.g. from a list of dictionaries) | `pd.DataFrame(records)` |
| `pd.read_csv(path, index_col=0)` | Load a CSV (file or URL), using column 0 as the index | `pd.read_csv(DATA_URL, index_col=0)` |
| `.head(n)` / `.tail(n)` | First / last `n` rows | `df.head()` |
| `.shape` / `.columns` | (rows, columns) / column names | `df.shape` |
| `.info()` | Columns, types, and non-missing counts | `df.info()` |
| `.describe()` | Summary statistics for numeric columns | `df.describe()` |
| `df["col"]` | One column, as a Series | `df["Rating"]` |
| `df[["a", "b"]]` | Several columns, as a DataFrame | `df[["Rating", "Age"]]` |
| `.iloc[pos]` | Select rows by **position** (end excluded) | `df.iloc[10:13]` |
| `.loc[label, col]` | Select by **label** (end included) | `df.loc[2, "Title"]` |
| `df[condition]` | Keep rows where the condition is `True` | `df[df["Rating"] == 1]` |
| `&`, `\|`, `.isin()` | Combine conditions / match a list | `df["Dept"].isin(["A", "B"])` |
| `.sort_values(col)` | Sort rows (`ascending=False` for largest first) | `df.sort_values("Age")` |
| `.value_counts()` | Count each value (`normalize=True` for shares) | `df["Rating"].value_counts()` |
| `.nunique()` / `.unique()` | Number of / list of distinct values | `df["Class Name"].nunique()` |

The whole lab in one sentence: **`read_csv` loads a table, `info` and `describe` tell you what's in it, and square brackets with column names, `loc`/`iloc`, and conditions let you pull out exactly the part you want to look at.**